In [2]:
# Independent additional homework solution. Standard library, no model/API.
import json

RECORDS = {'REF-1001': {'owner': 'demo-user-1', 'status': 'under_review'},
           'REF-1002': {'owner': 'demo-user-2', 'status': 'approved'}}


def dispatch(call, trusted_user):
    # A supplied tool-call object is untrusted data, never executable code.
    if not isinstance(call, dict) or set(call) != {'name', 'arguments'}:
        return {'code': 'invalid_call'}
    if call['name'] != 'check_status':
        return {'code': 'tool_not_allowed'}
    args = call['arguments']
    if not isinstance(args, dict) or set(args) != {'reference'}:
        return {'code': 'invalid_arguments'}
    ref = args['reference']
    if not isinstance(ref, str) or len(ref) != 8 or not ref.startswith('REF-') or not ref[4:].isascii() or not ref[4:].isdigit():
        return {'code': 'invalid_arguments'}
    record = RECORDS.get(ref)
    # Same outward result for absent and unauthorized records.
    if record is None or record['owner'] != trusted_user:
        return {'code': 'unavailable'}
    return {'code': 'ok', 'status': record['status']}


def run_tool_homework():
    before = json.dumps(RECORDS, sort_keys=True)
    cases = [
        ({'name': 'check_status', 'arguments': {'reference': 'REF-1001'}}, {'code': 'ok', 'status': 'under_review'}),
        ({'name': 'check_status', 'arguments': {'reference': 'REF-1002'}}, {'code': 'unavailable'}),
        ({'name': 'check_status', 'arguments': {'reference': 'REF-9999'}}, {'code': 'unavailable'}),
        ({'name': 'delete_record', 'arguments': {'reference': 'REF-1001'}}, {'code': 'tool_not_allowed'}),
        ({'name': 'check_status', 'arguments': {'reference': 'REF-1001', 'owner': 'demo-user-2'}}, {'code': 'invalid_arguments'}),
        ({'name': 'check_status', 'arguments': {'reference': 1001}}, {'code': 'invalid_arguments'}),
        ('not a tool object', {'code': 'invalid_call'}),
    ]
    rows = []
    for index, (call, expected) in enumerate(cases, 1):
        actual = dispatch(call, trusted_user='demo-user-1')
        assert actual == expected, (index, actual, expected)
        rows.append({'case': index, 'result': actual, 'passed': actual == expected})
    assert json.dumps(RECORDS, sort_keys=True) == before
    result = {'mode': 'simulated_tool_protocol', 'passed': len(rows), 'total': len(cases),
              'records_unchanged': True, 'cases': rows}
    print(json.dumps(result, indent=2))
    return result




In [3]:
result = run_tool_homework()


{
  "mode": "simulated_tool_protocol",
  "passed": 7,
  "total": 7,
  "records_unchanged": true,
  "cases": [
    {
      "case": 1,
      "result": {
        "code": "ok",
        "status": "under_review"
      },
      "passed": true
    },
    {
      "case": 2,
      "result": {
        "code": "unavailable"
      },
      "passed": true
    },
    {
      "case": 3,
      "result": {
        "code": "unavailable"
      },
      "passed": true
    },
    {
      "case": 4,
      "result": {
        "code": "tool_not_allowed"
      },
      "passed": true
    },
    {
      "case": 5,
      "result": {
        "code": "invalid_arguments"
      },
      "passed": true
    },
    {
      "case": 6,
      "result": {
        "code": "invalid_arguments"
      },
      "passed": true
    },
    {
      "case": 7,
      "result": {
        "code": "invalid_call"
      },
      "passed": true
    }
  ]
}


## Validation vs. authorization
1. Validation checks that the call has the correct shape and types. For example, case 5 was rejected with `invalid_arguments` because it included an extra `owner` field that is not allowed. Authorization checks that the current user actually owns the record. For example, case 2 was well-formed but rejected with `unavailable` because REF-1002 belongs to another user.
2. Limitations: this is a simulation only. `demo-user-1` is a fake identity with no real login or authentication, and I wrote the tool calls by hand, so no real AI model chose the tool.